In [1]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=0962ced0cc5b383de03fa9416375afb0dbaf1f3fdc441bcce59c32e4fc1a0b7f
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [2]:
import csv
import re
import pandas as pd
import nltk
import numpy as np
import joblib
from langdetect import detect, detect_langs, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from tqdm.auto import tqdm

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction import text
from sklearn.metrics import confusion_matrix

tqdm.pandas()
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [3]:
file_path = '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/dataset.csv'
df = pd.read_csv(file_path)

In [4]:
df.head()

,label,text
0,1,Congratulations! You've been selected for a lu...
1,1,URGENT: Your account has been compromised. Cli...
2,1,You've won a free iPhone! Claim your prize by ...
3,1,Act now and receive a 50% discount on all purc...
4,1,Important notice: Your subscription will expir...


In [5]:
#Language detection
DetectorFactory.seed = 0

def lang_detect(text):
  if not isinstance(text,str) or not text.strip():
    return 'Unknown'
  try:
    return detect(text)
  except LangDetectException:
    return 'Unknown'

In [6]:
#detect language
df['language'] = df['text'].progress_apply(lang_detect)
#filter for english
df = df[df['language'] == 'en'].copy()

  0%|          | 0/45155 [00:00<?, ?it/s]

In [7]:
#regex Cleaning
def regex_clean(text):
    if not isinstance(text, str): # Ensure text is a string
        return ''

    # 1. Lowercase the text
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 3. Remove URLs/Hyperlinks
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 4. Remove Email Addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # 5. Remove Numbers (often randomized in spam)
    text = re.sub(r'\d+', ' ', text)

    # 6. Remove Punctuation and Special Characters
    text = re.sub(r'[^\w\s]', ' ', text)

    # 7. Collapse multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function to the 'text' column to create 'cleaned_text'
df['cleaned_text'] = df['text'].progress_apply(regex_clean)

  0%|          | 0/41863 [00:00<?, ?it/s]

In [8]:
#Additional cleaning - drop duplicates
df.drop_duplicates(subset=['cleaned_text'], keep='first')
df.rename(columns={'text': 'raw_text'},inplace=True)

In [9]:
lemmatizer = WordNetLemmatizer()
#POS tagging
def get_wordnet_pos_from_tag(tag):
    if not tag:
        return wordnet.NOUN
    first_letter = tag[0].upper()
    tag_dict = {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "V": wordnet.VERB,
        "R": wordnet.ADV
    }
    return tag_dict.get(first_letter, wordnet.NOUN)

#Lemmatization
def lemmatize_text(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    word_tags = nltk.pos_tag(words) # Tags the whole sentence once

    lemmatized_words = [
        lemmatizer.lemmatize(word, get_wordnet_pos_from_tag(tag))
        for word, tag in word_tags
    ]
    return " ".join(lemmatized_words)

In [10]:
df['cleaned_lemmatized'] = df['cleaned_text'].progress_apply(lemmatize_text)

  0%|          | 0/41863 [00:00<?, ?it/s]

In [11]:
#exporting cleaned dataset for ease of use
#df.to_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/cleaned_dataset.csv', columns=['cleaned_lemmatized', 'label'] , index=False)

In [12]:
#import validation dataset
df_validation = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/validation_dataset.csv')

In [13]:
#applying same nlp pipeline for validation dataset
df_validation.rename(columns={'Email Text': 'raw_text', 'Email Type': 'label'},inplace=True)
df_validation['label'] = df_validation['label'].map({'Phishing Email': 1, 'Safe Email': 0})
df_validation['cleaned_text'] = df_validation['raw_text'].progress_apply(regex_clean)
df_validation['cleaned_lemmatized'] = df_validation['cleaned_text'].progress_apply(lemmatize_text)

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

In [14]:
df_validation.head()

,raw_text,label,cleaned_text,cleaned_lemmatized
0,"Dear Jordan, your subscription has been succes...",0,dear jordan your subscription has been success...,dear jordan your subscription have be successf...
1,"Dear Casey, thank you for your purchase. Your ...",0,dear casey thank you for your purchase your or...,dear casey thank you for your purchase your or...
2,Congratulations! You've won a $3000 gift card....,1,congratulations you ve won a gift card click h...,congratulation you ve win a gift card click he...
3,You have a new secure message from your bank. ...,1,you have a new secure message from your bank c...,you have a new secure message from your bank c...
4,Your package delivery is pending. Please provi...,1,your package delivery is pending please provid...,your package delivery be pending please provid...


In [15]:
#df_validation.to_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/cleaned_validation_dataset.csv', columns=['cleaned_lemmatized', 'label'] , index=False)

In [19]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------------------------------------------------
# STEP 1: Split data
# ---------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_lemmatized'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# ---------------------------------------------------
# STEP 2: Build TF-IDF features
# ---------------------------------------------------
custom_stop_words = list(text.ENGLISH_STOP_WORDS) + ['escapenumber', 'escapelong']

vectorizer = TfidfVectorizer(
    stop_words=custom_stop_words,
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# ---------------------------------------------------
# STEP 3: Tune Logistic Regression (C via CV)
# ---------------------------------------------------
base_model = LogisticRegression(max_iter=3000, solver='liblinear')

param_grid = {"C": [0.1, 0.5, 1, 2, 5]}

grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train_vectorized, y_train)
model = grid.best_estimator_

print("Best C:", grid.best_params_["C"])

# ---------------------------------------------------
# STEP 4: Predict (default 0.5 threshold)
# ---------------------------------------------------
predictions = model.predict(X_test_vectorized)

print("\n--- Logistic Regression Accuracy ---")
print(f"{accuracy_score(y_test, predictions) * 100:.2f}%")

print("\n--- Detailed Performance Report ---")
print(classification_report(y_test, predictions))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, predictions))

# ---------------------------------------------------
# STEP 5: Top spam words
# ---------------------------------------------------
words = vectorizer.get_feature_names_out()
spam_word_weights = model.coef_[0]
top_10_indices = np.argsort(spam_word_weights)[-10:]
top_10_words = [words[i] for i in top_10_indices]

print("\n--- Logistic Regression Top 10 Spam Words ---")
print(top_10_words[::-1])

# ---------------------------------------------------
# STEP 6: Validation set — same vectorizer + model
# ---------------------------------------------------
validation_text = df_validation['cleaned_lemmatized']
y_val = df_validation['label']
X_val_vectorized = vectorizer.transform(validation_text)
val_predictions = model.predict(X_val_vectorized)

Best C: 5

--- Logistic Regression Accuracy ---
97.54%

--- Detailed Performance Report ---
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      4191
           1       0.99      0.96      0.98      4182

    accuracy                           0.98      8373
   macro avg       0.98      0.98      0.98      8373
weighted avg       0.98      0.98      0.98      8373


--- Confusion Matrix ---
[[4142   49]
 [ 157 4025]]

--- Logistic Regression Top 10 Spam Words ---
['http', 'info', 'claim', 'uk', 'hk', 'mobile', 'viagra', 'woman', 'health', 'symbol']


In [17]:
#export model and vectorizer
#joblib.dump(model, '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/logistic_regression_model.pkl')
#joblib.dump(vectorizer, '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/tfidf_vectorizer.pkl')

['/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/tfidf_vectorizer.pkl']